# Optimizacion 2026-1 - Bin Packing & Set Covering (Gurobi)

**Universidad Nacional de Colombia** - Ingenieria de Sistemas y Computacion  
**Autores:** David Ramirez (jdramirezt@unal.edu.co), Jaisson Machado Bautista  
**Solver:** Gurobi 11+ (gurobipy)

This is the **Google Colab edition** of the project. It is self-contained: the two OR-Library
instances are embedded in base64, so no external files are needed. Run the cells top to bottom.

> **Colab <-> local `.py` switch.** Every Colab-only step (pip install, writing the embedded
> instances) is guarded by the `COLAB` flag in the next cell. Set `COLAB = False` to run the
> same code as a plain `.py` on your PC with an academic license, reading the instances from
> the local `data/` folder instead. Nothing else changes.


## 0. Environment & license switch

In Colab, `pip install gurobipy` ships the **size-limited** license (<= 2000 vars / 2000 constrs):
enough for the full Set Covering (200 vars) but **not** for the full 120-item Bin Packing
(14,520 binaries). Two ways to unlock the full BPP:

- **WLS Academic license** (free with your `@unal.edu.co`): paste the three WLS params below.
  Works inside Colab, no install.
- **Local academic Named-User license** (`grbgetkey`) when running as a local `.py` (`COLAB = False`).

`ITEM_LIMIT` keeps the BPP under the restricted cap when no full license is active.


In [ ]:
# === ENVIRONMENT TOGGLE =====================================================
# True  -> running in Google Colab (pip install + write embedded instances)
# False -> running as a local .py on your PC with an academic license,
#          reading the instances from the local data/ folder.
COLAB = True

# === LICENSE SWITCH =========================================================
# The full 120-item BPP needs 120*120 + 120 = 14,520 binaries, which exceeds
# the size-limited license. Choose how to run the BPP:
#   ITEM_LIMIT = 40    -> restricted license  (40*40 + 40 = 1,640 vars < 2000)
#   ITEM_LIMIT = None  -> full instance        (needs WLS or academic license)
ITEM_LIMIT = 40

# Optional WLS Academic credentials (leave as empty strings to skip).
# Get them at https://portal.gurobi.com -> WLS Manager. With valid values here,
# you can set ITEM_LIMIT = None and solve the complete 120-item instance in Colab.
WLS_ACCESSID = ""   # e.g. "xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx"
WLS_SECRET   = ""
WLS_LICENSEID = 0    # integer license id


In [ ]:
# Colab-only: install the solver. Skipped automatically when COLAB = False.
if COLAB:
    !pip install -q gurobipy


In [ ]:
import time, os, io, base64
import numpy as np
import matplotlib.pyplot as plt
import gurobipy as gp
from gurobipy import GRB
print('gurobipy', gp.gurobi.version())


## 1. Instances

In Colab the two OR-Library files are decoded from base64 into a local `data/` folder.
When `COLAB = False`, this cell does nothing and the loaders read your existing `data/` files.


In [ ]:
# Embedded OR-Library instances (base64). Only written to disk in Colab.
BPP_B64 = (
    "MQoxMjAgMTUwIDQ4CjM0CjIzCjU1CjUxCjQ4CjM3CjMzCjg5CjMxCjk1Cjc0CjI0CjIzCjMxCjQ3CjQ5Cjg0Cjk3CjIzCjkxCjQ1Cjg5CjczCjQ4Cjc3Cjk1CjU1CjIwCjQwCjc0CjYzCjU1CjM5CjQ3CjYzCjMzCjMxCjY4CjMyCjY1CjY0Cjk3CjUzCjI1Cjc4Cjg4CjM1CjY4CjMwCjkwCjU3CjEwMAo5OQo2Ngo5Mwo0NAoyOAoyNQo0OQo1NwozMAo0OQozMgo2OAo1NQo3OAo2Ngo0MAo2Nwo2NQo0Ngo1NAoyOQo5Nwo0MQo4OAo1MQo0MAo3OQo2OAo1NAo5MQo0OAo2MQoyNwo0OQoyNAo2MAo3MQo1NAoyOAo0Nwo5Mgo2MAo0Nwo4Mwo3MAo3OAozOAo1MwozNwo1MQo5MQo4OAo1Mwo5NAo3NAo5NAo3MQo2Ngo0OAozNwo4NQo4MwozMQoyNgozNAozOQoxMDAKNDAK"
)
SCP_B64 = (
    "NTAgMjAwCjg4IDU1IDc3IDkgNTAgNDkgNzcgNjAgNjggMzMKNzEgMiA4OCA5MyAxNSA4OCA2OSA5NyAzNSA5OQo4MyA0NCAxNSAzOCA1NiAyMSA1OSAxIDkzIDkzCjM0IDY1IDk4IDIzIDY1IDE0IDgxIDM5IDgyIDY1Cjc4IDI2IDIwIDQ4IDk4IDIxIDcwIDEwMCA2OCAxCjc3IDQyIDYzIDMgMTUgNDcgNDAgMzEgOCAzMQo3MyAxMSAxMSA5NCA2MyA5IDk4IDY5IDk5IDE3CjE3IDg1IDYxIDcxIDIyIDM0IDY4IDc4IDU1IDI4CjcwIDk3IDk0IDg5IDI2IDkyIDQwIDUyIDg2IDg0CjQ4IDU3IDY3IDU4IDE2IDMyIDI5IDkgNDQgMwo3NiA3MSAzMCA3NiAyOSAxIDEwIDkxIDgxIDgKMzAgOSA1IDQzIDEwIDY2IDMxIDM2IDg2IDYzCjI4IDcwIDE3IDkzIDc0IDc0IDYxIDMyIDYxIDUzCjI1IDEzIDEzIDg1IDU2IDQ2IDU1IDUzIDYwIDk0CjcgODcgODQgODMgMTMgOCA1MiA5NCA0NCAxNAozMiAyNSAyNSA2OSA1OCAxOCA1NSAyNCAzNiA2MAozMiAxMCA1NyA3MSAxMyA3IDg0IDcwIDIgMTIKOTcgMzEgMjIgNTMgNjMgNjIgMjggNTIgOCAyMgo0OSAxIDUwIDM0IDU5IDM3IDU1IDkwIDk0IDcyCjg1IDkyIDYzIDIwIDI1IDM4IDI4IDggNzUgOTUKMjkKNCA3IDEwIDExIDE5IDIyIDI2IDMzIDQxIDQ1CjQ5IDUxIDU3IDY1IDY4IDcwIDc0IDgwIDgzIDg5Cjk4IDk5IDExMSAxMTYgMTQ2IDE4MCAxOTUgMTk3IDE5OQozNwoxNCAxNiAxNyAyNSAzMiAzNCAzNSAzNyAzOCA0MAo0NCA0NiA2MSA2NyA3OCA5MCA5NiA5OSAxMDAgMTAyCjExMCAxMTIgMTIwIDEzMyAxNDEgMTQzIDE0NSAxNDcgMTQ4IDE1OAoxNjEgMTY0IDE3MiAxNzggMTg1IDE4NiAxODcKNDQKMyAxMiAxMyAxNSAxOCAxOSAyNSAzOCA0MSA1Nwo2MSA2NCA3MiA3MyA3NiA3OCA4NCA4NSA5MiA5NAoxMDEgMTE1IDExOCAxMjcgMTI4IDEzMCAxMzUgMTM2IDE0MiAxNDcKMTQ5IDE1MCAxNTEgMTU3IDE1OCAxNTkgMTcwIDE3MSAxODAgMTgzCjE4NyAxOTIgMTk2IDE5NwozOAoxIDkgMjggMzIgMzUgMzcgNTAgNTEgNTYgNTkKNjUgNjYgNzggODQgOTYgMTAwIDEwMiAxMDYgMTA3IDExMwoxMTYgMTI2IDEyNyAxMjkgMTMxIDEzOCAxNDAgMTQ0IDE1NyAxNjcKMTczIDE3NiAxNzkgMTgwIDE4NiAxODggMTk0IDE5Ngo0MwoyIDQgMTYgMjAgMjUgMzAgMzEgMzcgNDAgNDEKNTIgNTcgNTggNTkgNzAgNzUgNzYgODEgODcgOTEKOTcgMTAwIDEwMiAxMDMgMTA2IDExMiAxMTggMTE5IDEyMiAxMjQKMTI2IDEyNyAxMzEgMTM0IDE0NyAxNDggMTU3IDE1OCAxNjIgMTY1CjE2NiAxNzMgMTgxCjQ2CjEgMyA5IDI0IDMwIDM1IDM3IDQzIDQ1IDQ2CjUzIDU3IDYwIDYxIDYzIDY0IDY4IDc1IDc2IDc5Cjg1IDg5IDk2IDEwMiAxMDYgMTEyIDExMyAxMjAgMTIxIDEyNwoxMjkgMTMyIDEzMyAxMzkgMTQxIDE0MyAxNDcgMTU1IDE1NiAxNjIKMTY1IDE3MCAxNzEgMTc4IDE5MiAxOTYKNDIKNCA3IDggMTMgMTUgMTggMzkgNDAgNDEgNDkKNTEgNTYgNjQgNzQgNzcgODIgODQgOTAgOTQgOTUKMTAxIDEwMiAxMDQgMTA1IDEwNiAxMDkgMTIxIDEyMyAxMjkgMTMxCjEzNSAxMzggMTQxIDE0OCAxNTYgMTc1IDE3NiAxODAgMTgyIDE5MwoxOTQgMTk1CjQ0CjIgOCAxMSAxNyAxOCAyMCAyOCAzOCAzOSA0MAo0NyA0OCA1MSA1OSA2MCA2NSA2OCA4NyA4OSA5Mgo5NiA5OSAxMDAgMTA0IDEwNyAxMDkgMTE1IDExNiAxMjAgMTIyCjEzOSAxNDQgMTQ1IDE1MyAxNjEgMTYyIDE2MyAxNjcgMTc1IDE4MgoxODUgMTg3IDE5MSAxOTkKNDkKNCA1IDEwIDEzIDIxIDI4IDM0IDM1IDM4IDQ5CjUwIDY2IDcwIDczIDg0IDg1IDg2IDkyIDkzIDk4Cjk5IDEwMCAxMDMgMTA2IDExNiAxMTcgMTMyIDEzMyAxMzQgMTM1CjEzOCAxNDIgMTQ5IDE1OCAxNjAgMTYyIDE2NSAxNjYgMTY3IDE2OQoxNzQgMTgwIDE4MiAxODMgMTg0IDE4OCAxOTMgMTk1IDE5OQo0NAo4IDEyIDEzIDE0IDI1IDI2IDMxIDM4IDQyIDUzCjU5IDY5IDczIDc0IDc1IDc4IDgwIDk0IDk4IDEwMwoxMDYgMTA4IDExMSAxMTIgMTEzIDEyMyAxMjUgMTMyIDEzNCAxMzYKMTM5IDE0MiAxNDMgMTQ2IDE1OCAxNTkgMTY3IDE3MCAxNzQgMTgzCjE4NCAxODYgMTg3IDE5MQozNQoxIDYgMTEgMTQgMTUgMjQgMjkgMzQgMzkgNTcKNTkgNjQgNjcgNzcgNzkgODEgMTEyIDExMyAxMjAgMTIyCjEzMCAxMzYgMTM4IDEzOSAxNTYgMTU4IDE2MyAxNjggMTcwIDE3MwoxNzcgMTg0IDE4NiAxOTUgMTk5CjM2CjE0IDE3IDIyIDI3IDM1IDQyIDQ0IDQ4IDQ5IDUxCjU3IDYyIDYzIDc5IDk5IDEwMCAxMTAgMTExIDExNyAxMjMKMTI0IDEzMCAxMzQgMTM2IDE0NiAxNDkgMTU0IDE1NSAxNjEgMTYyCjE2MyAxNjcgMTk2IDE5NyAxOTkgMjAwCjM5CjE1IDE2IDE4IDE5IDIxIDI2IDM4IDQwIDQ1IDQ3CjQ4IDQ5IDU5IDY1IDY5IDcyIDgyIDg0IDg2IDk2CjEyMiAxMjMgMTI4IDEzNCAxMzcgMTM4IDE1MCAxNTIgMTYyIDE2MwoxNjkgMTc3IDE4MyAxODcgMTkwIDE5MyAxOTUgMTk5IDIwMAozMAozIDUgOSAxMyAyMyAyNCAzMSAzMiAzNiAzOAo0NCA0OCA1NiA2NiA5NiA5NyAxMDQgMTA4IDExMiAxMTQKMTE2IDEzNCAxNDMgMTYxIDE3MiAxNzQgMTc4IDE4NyAxODggMTkwCjM2CjE1IDI0IDI3IDMxIDMzIDM3IDQ2IDQ3IDQ5IDU3CjU4IDU5IDYxIDY4IDcyIDc2IDgwIDg0IDg4IDkwCjEwMSAxMDUgMTA5IDExNSAxMTggMTIxIDEzNCAxMzcgMTM5IDE0MQoxNjMgMTY4IDE4NiAxOTAgMTk0IDE5Ngo0OAoyIDMgNCA2IDEzIDE0IDE1IDI0IDI1IDI2CjI4IDI5IDMwIDM4IDQwIDQzIDQ1IDQ2IDUzIDU3CjY3IDY5IDcyIDc2IDgwIDgyIDkzIDk1IDk2IDk5CjEyMSAxMjMgMTMwIDEzNiAxMzcgMTQ0IDE0NiAxNDggMTU4IDE2MQoxNjQgMTczIDE3NSAxNzYgMTc4IDE4NyAxOTAgMTk0CjM3CjMgNCA1IDggOSAxNyAxOCAyOSA0NCA0Nwo1MSA1NiA1OCA2MSA2NCA3NSA4MCA4MSA4NCA4Nwo4OSAxMDAgMTA2IDExNiAxMjMgMTMyIDEzOCAxNDMgMTQ4IDE1MQoxNjEgMTY5IDE4MCAxODYgMTkwIDE5MiAxOTkKNDMKOCA5IDE1IDE2IDI5IDMwIDMzIDM2IDQyIDQzCjQ0IDQ1IDQ3IDQ4IDQ5IDU5IDYwIDYxIDYzIDY2CjY5IDcwIDgzIDg2IDk0IDk2IDk4IDEwMyAxMDkgMTEwCjExNCAxMjYgMTM5IDE1MyAxNTQgMTU3IDE2MSAxNjkgMTc1IDE5MAoxOTEgMTkzIDE5OQo1Ngo2IDggMjEgMjQgMzAgNDYgNDkgNTAgNTEgNTcKNTggNjAgNjQgNjYgNjkgNzAgNzYgNzggODEgODIKODQgODUgODcgMTA0IDEwNyAxMTUgMTE2IDEyMCAxMjcgMTMwCjEzMiAxMzYgMTM3IDEzOSAxNDIgMTQ0IDE0NiAxNTAgMTUyIDE1NgoxNjEgMTYzIDE2OCAxNzMgMTc2IDE3OSAxODAgMTgxIDE5MCAxOTEKMTkyIDE5NSAxOTYgMTk3IDE5OCAyMDAKNDcKNCA3IDEzIDE1IDIwIDIyIDI0IDMzIDM5IDQwCjQ2IDQ3IDQ5IDUzIDU1IDU5IDYwIDcwIDgxIDgyCjkxIDkyIDEwMSAxMDcgMTExIDExMiAxMTUgMTE2IDEyMSAxMjYKMTMyIDEzMyAxMzQgMTM3IDE0NCAxNTEgMTU2IDE2MyAxNjUgMTY3CjE3MCAxNzMgMTc4IDE4NCAxODUgMTkwIDE5Nwo0MQoxIDMgNCAxOCAyMCAyMSAyMyAyOCAzMCAzNgo0MyA0NCA2MSA2MyA2NCA3MiA3NiA3OCA4NSA5MAoxMDMgMTA1IDExNiAxMTkgMTIwIDEyNSAxMjggMTQxIDE1MSAxNTYKMTYwIDE2MiAxNjYgMTY4IDE3NCAxNzUgMTc3IDE4NCAxODcgMTg4CjE5MAo0Nwo5IDEwIDE0IDE2IDE3IDIwIDI0IDMwIDMyIDM0CjM2IDQxIDQ0IDQ3IDUxIDYzIDY0IDcwIDcxIDczCjc1IDg3IDk2IDk4IDEwNSAxMDcgMTExIDExMiAxMjEgMTI5CjEzMSAxMzIgMTMzIDEzNCAxMzUgMTQ5IDE1MyAxNTQgMTcwIDE3NAoxODIgMTgzIDE4NSAxODkgMTk0IDE5OCAxOTkKMzkKNSAxMyAxNSAxOCAzMyAzNyA0MSA0NCA0NyA2Mwo3MSA3OCA4MSA4MiA4NCA5NiA5OSAxMDcgMTA5IDExMQoxMTUgMTE4IDEyMiAxMjYgMTMyIDEzOSAxNDYgMTQ4IDE0OSAxNTIKMTU3IDE2MiAxNjcgMTY4IDE3NCAxNzggMTg5IDE5MSAxOTIKNDMKNiAxMiAxMyAzOSA0MSA0MiA0NiA0OSA2NiA3OAo4NSA4NiA4OSA5MCA5MyA5NCA5NSAxMDEgMTA2IDEwOQoxMTIgMTIwIDEyNyAxMzAgMTMzIDEzNiAxMzcgMTM4IDE0MCAxNDEKMTQ3IDE1MyAxNTQgMTU4IDE1OSAxNjAgMTYzIDE2NCAxODAgMTgxCjE4MiAxODMgMTg1CjM2CjE1IDE4IDIyIDI2IDMxIDMyIDMzIDM0IDM2IDQwCjQ0IDU1IDY5IDcyIDczIDc2IDgwIDgyIDkzIDExMgoxMjcgMTMwIDEzMiAxMzMgMTM2IDEzNyAxNDEgMTQ3IDE1NCAxNTkKMTYxIDE3NiAxODEgMTg2IDE4OCAxOTEKNDMKMiA0IDE2IDE3IDIxIDIyIDI2IDM0IDM1IDQwCjQ2IDUyIDYwIDY0IDY1IDc1IDgzIDg4IDg5IDk2CjEwMiAxMDUgMTEyIDExOCAxMjAgMTIxIDEzMiAxNDAgMTUxIDE1NAoxNTggMTYwIDE2NSAxNzMgMTc0IDE3OCAxODAgMTgyIDE4MyAxODQKMTkxIDE5NyAyMDAKNDAKMyAxNCAyMSAzMiAzMyAzNiA0MSA0NSA1NCA1Nwo2MyA2NCA2NSA2OCA2OSA3MCA3MiA3OSA4NSA5MQo5NCAxMDAgMTA1IDEwNyAxMTIgMTE0IDEyMCAxMzAgMTM2IDEzOAoxNDAgMTQ0IDE0NiAxNjUgMTcyIDE3NiAxODUgMTg3IDE5NCAxOTUKNDIKOSAxMSAxMyAxOCAxOSAyMCAyMyAyNyAyOSAzNQo0MSA0MiA0MyA0NyA1NSA1OSA2MSA2MyA3NiA3OAo4MSAxMDIgMTA0IDEwNSAxMDYgMTA3IDExNCAxMTUgMTE3IDExOAoxMTkgMTIxIDEzMyAxMzkgMTQ0IDE0NiAxNTUgMTU2IDE1NyAxNjcKMTgwIDIwMAo0NAo2IDExIDIzIDI5IDMwIDQzIDUxIDU1IDU3IDYwCjY2IDcxIDc0IDc3IDgxIDg1IDk1IDEwMSAxMDIgMTA1CjExNCAxMjUgMTI2IDEyNyAxMjggMTI5IDEzMCAxMzYgMTM3IDEzOQoxNDIgMTU0IDE1NiAxNTcgMTU4IDE1OSAxNjEgMTYyIDE2NiAxNzMKMTc1IDE4MSAxODcgMTkyCjQ1CjQgMTUgMjMgMjUgMjggMzIgMzQgMzUgMzYgMzgKNDIgNDMgNDcgNTggNjAgNjQgNjUgNjggNjkgNzIKNzQgODAgODcgODkgOTEgOTMgOTYgMTA0IDEwNSAxMTIKMTE5IDEyMCAxMzcgMTQ0IDE0OSAxNTAgMTY2IDE3MyAxNzQgMTc1CjE3OCAxODQgMTg5IDE5OCAxOTkKNDQKMSAxNSAyNCAyNSAyOSAzMyAzNyAzOCA0MyA0NQo0NiA0OCA1MSA1NyA2MiA2MyA2OSA3MSA3NSA4Mgo4NyA5MCAxMDAgMTAyIDEwNiAxMDggMTI2IDEyOCAxMzcgMTM4CjE0MiAxNDcgMTQ5IDE1MSAxNjUgMTczIDE3NCAxODAgMTg1IDE4NgoxODggMTkwIDE5MiAxOTUKMzkKOSAyNiAzMCAzMyAzNCA0MSA0NCA0NSA1MSA1Mgo2MCA2MyA2NCA2NiA2OSA3MCA3MyA3NiA4MCA4NAo5MiA5NCAxMDIgMTE0IDExNSAxMTcgMTI0IDEzNCAxNDUgMTU0CjE2MSAxNjQgMTcyIDE3NyAxNzggMTc5IDE4NSAxOTAgMTk3CjQzCjEgNSA5IDE3IDE4IDIwIDI0IDI5IDQxIDYwCjY1IDY5IDcwIDczIDg2IDEwMSAxMDQgMTA4IDExOCAxMjIKMTI2IDEyNyAxMjggMTI5IDEzMyAxMzUgMTM5IDE0MCAxNDcgMTUxCjE1OCAxNjMgMTY3IDE3MCAxNzUgMTc4IDE4MSAxODMgMTg0IDE4NQoxODcgMTk5IDIwMAozNwoxIDMgNyAxOSAyNyAyOCA0MiA0NSA1NSA2Mgo3NCA3OSA4NiAxMDAgMTAxIDEwNCAxMTIgMTE1IDEyMSAxMjIKMTMwIDEzMyAxMzUgMTM4IDE0NCAxNDcgMTU0IDE2MCAxNjEgMTY1CjE2NyAxNjggMTg0IDE4NiAxODggMTk0IDE5OAozOQo1IDYgMTIgMTcgMTkgMzAgMzIgMzMgMzcgNDEKNDggNDkgNTkgNjAgNjUgNzEgNzYgODEgODYgODkKOTMgOTQgOTUgOTggMTE4IDEzNSAxMzcgMTQ1IDE0NyAxNDkKMTUxIDE3MCAxODAgMTg1IDE4OSAxOTEgMTkzIDE5NiAxOTgKNDgKNyA4IDExIDEzIDIxIDIyIDI4IDI5IDQ1IDQ2CjQ3IDUwIDU0IDU3IDY0IDY2IDY4IDY5IDcxIDg4CjkwIDk3IDEwMyAxMDQgMTA2IDEwNyAxMDkgMTEwIDExOCAxMjgKMTMxIDEzMyAxMzggMTQ3IDE2MSAxNjIgMTY0IDE2NSAxNjcgMTY5CjE3NSAxODAgMTgxIDE4OCAxOTAgMTkxIDE5MyAxOTUKNDIKMyAxOCAyMiAyNiAzNSAzOCA0MCA1MSA1MiA1Mwo1NiA2MCA2MSA3MiA3OSA4MiA4MyA5NyAxMDggMTA5CjExMSAxMTMgMTE5IDEyNSAxMjggMTM3IDE0NiAxNTkgMTYwIDE2NAoxNjUgMTcwIDE3MSAxNzIgMTc3IDE4NSAxODYgMTg3IDE5MSAxOTYKMTk3IDIwMAozNQoxIDMgMTIgMjMgMzIgMzUgNDAgNDMgNDYgNDgKNTUgNjEgNjMgNzAgNzMgODAgODEgODUgODkgOTQKOTcgOTkgMTAxIDEwMyAxMTUgMTIxIDEyNSAxMjYgMTI4IDEzMgoxNjcgMTcwIDE3OCAxODYgMTk0CjQzCjIgMyAxOCAyMyAyOCAzMyAzOSA0OSA1NCA1Nwo2MSA2NyA4NCA5NiA5OCAxMDMgMTA0IDEwNyAxMTAgMTEzCjExNCAxMTYgMTIzIDEyNyAxMjggMTMxIDEzMiAxMzggMTQ0IDE0NQoxNDcgMTU1IDE1NiAxNTggMTU5IDE3MCAxNzQgMTc5IDE4MyAxODQKMTg2IDE4NyAxOTcKNDQKMyA3IDE0IDIwIDIyIDI0IDI1IDI5IDMzIDM0CjM4IDQyIDQ0IDU0IDU2IDY2IDc1IDc2IDgwIDgyCjkwIDk0IDEwMSAxMDMgMTEwIDExMyAxMjAgMTI1IDEyNyAxMzAKMTM1IDEzNiAxNDIgMTQ2IDE1NyAxNjEgMTY1IDE3MCAxNzIgMTc2CjE4MiAxODQgMTkwIDIwMAo0OAo0IDkgMTAgMTUgMjQgMjYgMzAgNDAgNDIgNDUKNTIgNTYgNTggNjEgNjMgNjYgNjcgNzYgNzcgODAKODEgOTAgOTEgOTYgMTAzIDExMiAxMjAgMTIyIDEyNCAxMjUKMTI2IDEzMSAxNDUgMTQ4IDE0OSAxNTcgMTU4IDE2MyAxNjUgMTY3CjE2OSAxNzIgMTgwIDE4MyAxODcgMTg4IDE5NCAyMDAKNDYKNCA3IDE3IDIzIDM1IDM3IDQxIDQyIDU5IDYyCjczIDc1IDgwIDkxIDk0IDEwMiAxMDUgMTEwIDEyMyAxMjQKMTI1IDEyNyAxMjggMTMwIDEzMSAxMzMgMTM4IDE0MSAxNDkgMTUyCjE1MyAxNTYgMTY2IDE3MCAxNzEgMTcyIDE3MyAxNzkgMTgwIDE4MwoxODYgMTkwIDE5MiAxOTUgMTk2IDE5OAo0NgoxIDMgNCA3IDEzIDE2IDE5IDIwIDIxIDIyCjIzIDI0IDI4IDMzIDM0IDM1IDQxIDQzIDQ0IDQ1CjUxIDU5IDY0IDY1IDY5IDczIDgyIDgzIDkwIDk5CjEwNiAxMDcgMTA4IDExNCAxMTggMTI1IDEzMiAxMzUgMTM2IDE0NwoxNDggMTU2IDE1OSAxNzAgMTk0IDE5NQo0NAoyIDkgMTQgMTcgMTkgMjIgMzQgNDAgNTEgNTIKNTggNjEgNjggNzkgODAgODEgODQgODggOTAgOTQKOTUgMTAyIDEwNiAxMDcgMTEwIDExNSAxMjUgMTMwIDEzMSAxMzIKMTQ0IDE1NCAxNTggMTU5IDE3MCAxNzMgMTc0IDE4MiAxODQgMTg1CjE4OCAxODkgMTk1IDIwMAozOQoxMiAxNCAxNSAxNiAyMSAyNCAyNiAyNyAyOSAzMQo0MyA0NCA0NSA0NiA0OCA1MCA1NiA1OCA2NCA3Mgo3NSA5MSA5NCAxMTEgMTE1IDEzMSAxMzYgMTM5IDE0MiAxNTkKMTYzIDE3MiAxODMgMTg2IDE4NyAxODkgMTkwIDE5MyAxOTQKNDAKMSA0IDcgOSAxMSAxOCAxOSAyNiAyOCAzMgo0MiA1OCA2NiA2NyA2OSA3MCA3MyA3NiA3NyA3OQo5NyAxMTMgMTE4IDEyMCAxMjEgMTI2IDEyOSAxMzAgMTMzIDE0MgoxNDQgMTQ2IDE1MCAxNjIgMTYzIDE2OCAxNzUgMTg4IDE5MiAxOTQKMzcKMSAxMSAxMyAxOCAyOSAzMCAzOCAzOSA0NCA1NAo1NSA2MCA4NCA4OCA4OSA5NCA5NSAxMDYgMTA3IDExMAoxMTMgMTE2IDEyMiAxMjMgMTMwIDEzMiAxNDQgMTQ4IDE0OSAxNTUKMTU3IDE2MiAxNjMgMTY0IDE3NyAxODggMTkxCjQyCjEgMTEgMTQgMTUgMjAgMjIgMzAgMzggMzkgNDMKNDUgNDcgNDggNjAgNjEgNjYgNzAgODIgODQgODgKOTQgMTAxIDEwNSAxMTcgMTIzIDEyOCAxMzYgMTM5IDE0NyAxNDgKMTQ5IDE1MCAxNTQgMTU4IDE2MSAxNjcgMTczIDE4NiAxODggMTkzCjE5NCAxOTkKNDMKMyA0IDkgMjQgMjggMzYgMzggNDAgNDkgNTEKNTQgNTYgNTcgNjIgNzEgNzIgNzMgNzcgODAgODgKOTIgOTYgOTcgOTggMTA2IDExMCAxMTEgMTI0IDEyNSAxMzMKMTQyIDE0NiAxNDkgMTUyIDE1NSAxNTYgMTc4IDE4MSAxODMgMTg1CjE4OSAxOTYgMTk3CjQ5CjEgMiA5IDEwIDI3IDMxIDMyIDMzIDM1IDM2CjM5IDQzIDQ3IDU1IDU5IDY4IDY5IDcwIDc1IDc2Cjg0IDkxIDk1IDEwMSAxMDQgMTA3IDExMCAxMTMgMTE0IDExNQoxMjAgMTIyIDEyNCAxMjUgMTI2IDE0NiAxNDcgMTQ5IDE1NiAxNjcKMTcxIDE3NyAxNzggMTgyIDE4NSAxODggMTkxIDE5NCAxOTcK"
)

DATA_DIR = 'data'
BPP_PATH = os.path.join(DATA_DIR, 'BinPacking_u120_01.txt')
SCP_PATH = os.path.join(DATA_DIR, 'SetCovering_50x200.txt')

if COLAB:
    os.makedirs(DATA_DIR, exist_ok=True)
    with open(BPP_PATH, 'wb') as f:
        f.write(base64.b64decode(BPP_B64))
    with open(SCP_PATH, 'wb') as f:
        f.write(base64.b64decode(SCP_B64))
    print('Instances written to', DATA_DIR)
else:
    # Local mode: assume the repo layout (this notebook lives in notebooks/).
    print('Local mode: reading instances from', DATA_DIR)


In [ ]:
# --- OR-Library parsers -----------------------------------------------------
def load_bin_packing(filepath):
    """Parse OR-Library BPP file -> (n_items, capacity, weights)."""
    with open(filepath) as f:
        lines = [ln.strip() for ln in f if ln.strip()]
    idx = 1                                   # line 0 = number of test cases
    header = lines[idx].split(); idx += 1
    n_items, capacity = int(header[0]), int(header[1])
    weights = [int(lines[idx + k]) for k in range(n_items)]
    return n_items, capacity, weights

def load_set_covering(filepath):
    """Parse OR-Library SCP file -> (m_rows, n_cols, costs, coverage[i]=cols)."""
    with open(filepath) as f:
        tok = f.read().split()
    i = 0
    m = int(tok[i]); i += 1
    n = int(tok[i]); i += 1
    costs = [int(tok[i + j]) for j in range(n)]; i += n
    coverage = []
    for _ in range(m):
        k = int(tok[i]); i += 1
        cols = [int(tok[i + t]) - 1 for t in range(k)]; i += k   # to 0-based
        coverage.append(cols)
    return m, n, costs, coverage


## 2. Bin Packing Problem (BPP)

$$\min \sum_{j} y_j \quad\text{s.t.}\quad \sum_j x_{ij}=1\ \forall i,\quad \sum_i w_i x_{ij} \le C\,y_j\ \forall j,\quad x_{ij},y_j\in\{0,1\}$$

`x[i,j]=1` if item *i* goes to bin *j*; `y[j]=1` if bin *j* is used. The capacity row also links
`x` to `y`. A symmetry-breaking row `y[j] >= y[j+1]` removes equivalent bin relabelings.


In [ ]:
def solve_bin_packing(data_path, item_limit=None, time_limit=300, verbose=True):
    n, C, w = load_bin_packing(data_path)
    n_full = n
    trial = item_limit is not None and item_limit < n
    if trial:
        n, w = item_limit, w[:item_limit]      # restricted-license reduction
    I = J = list(range(n))

    model = gp.Model('BinPacking')
    model.Params.LogToConsole = 1 if verbose else 0
    model.Params.TimeLimit = time_limit

    x = model.addVars(I, J, vtype=GRB.BINARY, name='x')   # item -> bin
    y = model.addVars(J, vtype=GRB.BINARY, name='y')      # bin used?
    model.setObjective(gp.quicksum(y[j] for j in J), GRB.MINIMIZE)
    model.addConstrs((gp.quicksum(x[i, j] for j in J) == 1 for i in I), name='assign')
    model.addConstrs((gp.quicksum(w[i] * x[i, j] for i in I) <= C * y[j] for j in J), name='capacity')
    model.addConstrs((y[j] >= y[j + 1] for j in range(n - 1)), name='symmetry')

    t0 = time.time(); model.optimize(); dt = time.time() - t0
    bins = [[i for i in I if x[i, j].X > 0.5] for j in J if y[j].X > 0.5]
    out = dict(n=n, n_full=n_full, C=C, w=w, trial=trial, bins=bins,
               used=len(bins), gap=model.MIPGap, secs=dt, status=model.Status)
    if verbose:
        print(f"\nBins used: {out['used']}  gap: {out['gap']:.4%}  time: {dt:.2f}s"
              + (f"  [restricted: {n}/{n_full} items]" if trial else ''))
    return out

bpp = solve_bin_packing(BPP_PATH, item_limit=ITEM_LIMIT)


## 3. Set Covering Problem (SCP)

$$\min \sum_j c_j x_j \quad\text{s.t.}\quad \sum_{j \in A(i)} x_j \ge 1\ \forall i,\quad x_j\in\{0,1\}$$

`x[j]=1` if column *j* is chosen; every row *i* must be covered by at least one chosen column.


In [ ]:
def solve_set_covering(data_path, time_limit=300, verbose=True):
    m, n, costs, cover = load_set_covering(data_path)
    model = gp.Model('SetCovering')
    model.Params.LogToConsole = 1 if verbose else 0
    model.Params.TimeLimit = time_limit

    x = model.addVars(range(n), vtype=GRB.BINARY, name='x')
    model.setObjective(gp.quicksum(costs[j] * x[j] for j in range(n)), GRB.MINIMIZE)
    model.addConstrs((gp.quicksum(x[j] for j in cover[i]) >= 1 for i in range(m)), name='cover')

    t0 = time.time(); model.optimize(); dt = time.time() - t0
    chosen = [j for j in range(n) if x[j].X > 0.5]
    out = dict(m=m, n=n, costs=costs, cover=cover, chosen=chosen,
               cost=model.ObjVal, gap=model.MIPGap, secs=dt, status=model.Status)
    if verbose:
        print(f"\nTotal cost: {out['cost']:.0f}  columns: {len(chosen)}  "
              f"gap: {out['gap']:.4%}  time: {dt:.2f}s")
    return out

scp = solve_set_covering(SCP_PATH)


## 4. Sensitivity analysis (BPP capacity)

Sweep the bin capacity `C` from 100 to 200 (step 10) and record bins used, solve time and MIP gap.


In [ ]:
def solve_bpp_capacity(n, w, C, time_limit=120):
    I = J = list(range(n))
    model = gp.Model('BPP_sens'); model.Params.LogToConsole = 0; model.Params.TimeLimit = time_limit
    x = model.addVars(I, J, vtype=GRB.BINARY, name='x'); y = model.addVars(J, vtype=GRB.BINARY, name='y')
    model.setObjective(gp.quicksum(y[j] for j in J), GRB.MINIMIZE)
    model.addConstrs((gp.quicksum(x[i, j] for j in J) == 1 for i in I), name='assign')
    model.addConstrs((gp.quicksum(w[i] * x[i, j] for i in I) <= C * y[j] for j in J), name='capacity')
    model.addConstrs((y[j] >= y[j + 1] for j in range(n - 1)), name='symmetry')
    t0 = time.time(); model.optimize(); dt = time.time() - t0
    return dict(C=C, used=int(round(model.ObjVal)), secs=dt, gap=model.MIPGap)

# Reuse the (possibly reduced) item set from the BPP run above.
sens = [solve_bpp_capacity(bpp['n'], bpp['w'], C) for C in range(100, 201, 10)]
for r in sens:
    print(f"C={r['C']:>3}  bins={r['used']:>2}  time={r['secs']:.2f}s")


In [ ]:
caps = [r['C'] for r in sens]; used = [r['used'] for r in sens]; secs = [r['secs'] for r in sens]
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(caps, used, '-o', color='steelblue', lw=2); ax[0].set_title('Bins used vs capacity')
ax[0].set_xlabel('Capacity C'); ax[0].set_ylabel('Bins used'); ax[0].grid(ls='--', alpha=.5)
ax[1].bar(caps, secs, width=7, color='darkorange'); ax[1].set_title('Solve time vs capacity')
ax[1].set_xlabel('Capacity C'); ax[1].set_ylabel('Seconds'); ax[1].grid(axis='y', ls='--', alpha=.5)
plt.tight_layout(); plt.show()


## 5. Quick visual summary (dashboard-style)

A compact in-notebook view of both solutions. The repo also ships a standalone, no-server
`dashboard/dashboard.html` with the same three panels (Bin Packing, Set Covering, Sensitivity).


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
# BPP: one horizontal bar per bin, width = load
loads = [sum(bpp['w'][i] for i in b) for b in bpp['bins']]
ax[0].barh(range(len(loads)), loads, color='mediumseagreen')
ax[0].axvline(bpp['C'], color='crimson', ls='--', label=f"capacity {bpp['C']}")
ax[0].set_title(f"Bin Packing - {bpp['used']} bins"); ax[0].set_xlabel('load'); ax[0].set_ylabel('bin'); ax[0].legend()
# SCP: coverage matrix with chosen columns highlighted
M = np.zeros((scp['m'], scp['n']))
for i, cols in enumerate(scp['cover']):
    for c in cols: M[i, c] = 1
ax[1].imshow(M, aspect='auto', cmap='Greys', alpha=.3)
for c in scp['chosen']: ax[1].axvline(c, color='dodgerblue', alpha=.6)
ax[1].set_title(f"Set Covering - cost {scp['cost']:.0f}, {len(scp['chosen'])} cols")
ax[1].set_xlabel('columns'); ax[1].set_ylabel('rows')
plt.tight_layout(); plt.show()


## 6. Notebook <-> local `.py` notes

- **Run in Colab (default):** keep `COLAB = True`. For the full 120-item BPP, fill the three
  `WLS_*` fields and set `ITEM_LIMIT = None`.
- **Run as a local `.py`:** `File -> Download -> .py`, set `COLAB = False`, drop the
  `!pip install` cell, and run from the repo root with your academic license active
  (`grbgetkey ...`). The loaders already read from `data/`.
- The standalone scripts `models/bin_packing.py`, `models/set_covering.py` and
  `sensitivity/sensitivity_analysis.py` mirror this notebook one-to-one.
